# Notebook 06 — Genie Code and Custom Skills

**What you'll do:**
1. Understand how **Genie Code skills** work — reusable domain knowledge in a markdown file
2. Copy the workshop's skill file into your `.assistant/skills/` folder (manual step)
3. Use a **specific prompt** to have Genie Code build a fully configured Genie SQL space programmatically

**Why this matters:** Genie SQL answers questions from your tables. **Genie Code** generates executable notebooks. A **skill** gives it domain knowledge, and a **prompt** tells it exactly what to build. Together, you can programmatically create a Genie space with instructions, sample questions, and curated Q→SQL — all from a single prompt.

**Before you start:** Run notebooks **02** (spaces) and **03** (benchmarks).

**Compute:** Serverless.

**References:**
- [What is Genie Code?](https://docs.databricks.com/en/genie/code/index.html)
- [Genie Code skills](https://docs.databricks.com/en/genie/code/skills.html)
- [AI/BI Genie overview](https://docs.databricks.com/en/genie/index.html)
- [Genie spaces REST API](https://docs.databricks.com/api/workspace/genie)

## Load workshop config


## Initialize workspace client


In [ ]:
%run ./00_workshop_config

In [ ]:
import re
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
host = w.config.host.rstrip("/")

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
PRIMARY_SPACE_ID = _cfg.get("genie_space_id", "")
if not PRIMARY_SPACE_ID:
    raise RuntimeError("genie_space_id not found in workshop_config. Run notebook 02 first.")
print(f"Primary space: {genie_ui_room_url(PRIMARY_SPACE_ID)}")

## Step 1: Understanding Genie Code vs Genie SQL

| Feature | Genie SQL (notebooks 02-05) | Genie Code (this notebook) |
|---------|---------------------------|---------------------------|
| **What it generates** | SQL queries against your tables | Python/SQL notebooks you can execute |
| **Where it lives** | AI/BI Genie space (web UI) | Databricks Assistant / notebook sidebar |
| **How to teach it** | Instructions + curated Q→SQL in space config | `.assistant/skills/` markdown files |
| **Best for** | Ad-hoc data questions, dashboards | Complex analysis, multi-step workflows, building spaces |

**Key insight:** The **skill** is generic domain knowledge (what OEE means, how manufacturing data is structured). The **prompt** is specific to YOUR tables and tells Genie Code what to build.

**Docs:** [What is Genie Code?](https://docs.databricks.com/en/genie/code/index.html) | [Skills reference](https://docs.databricks.com/en/genie/code/skills.html)

## Step 2: Review the skill file

The skill ships with this workshop in the **`skill/`** folder, right alongside your notebooks:

```
GM-Genie-Workshop/
  skill/
    manufacturing-analytics_genie/
      SKILL.md              <-- this is the skill file
  notebooks/
    00_workshop_config
    ...
    06_genie_code_skills    <-- you are here
```

Open `skill/manufacturing-analytics_genie/SKILL.md` in the workspace file browser to read it. It has **two sections:**

**Section 1 — Manufacturing Domain Knowledge**
- Metrics: OEE, FPY, defect rate, scrap rate, downtime
- Typical table structures and join paths
- Event type conventions, date handling, sensitive columns
- Analysis patterns (OEE, defects, shifts, safety)

**Section 2 — Genie Space Best Practices**
- How to write SQL instructions
- Sample questions (3-5 starters)
- Curated Q→SQL examples (8-12 teaching pairs)
- Benchmarks/Evals (12-16 ground-truth questions for scoring)
- Table metadata and column comments
- A/B testing pattern
- Monitoring and continuous improvement loop
- CI/CD export → transform → deploy pattern

The skill is **generic** — it teaches Genie Code about manufacturing AND about what a good Genie space looks like. It does NOT reference your specific catalog, schema, or table names.

In [ ]:
# Print where the skill file lives so you can find it in the workspace file browser
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
parent_dir = "/".join(notebook_path.split("/")[:-1])
skill_source = f"{parent_dir}/skill/manufacturing-analytics_genie/SKILL.md"

print("Skill file location (in your workspace):")
print(f"  {skill_source}")
print()
print("Open it in the workspace file browser to review the contents.")
print("You will copy this file in the next step.")

## Step 3: Copy the skill into `.assistant/skills/`

Genie Code discovers skills from the **`.assistant/skills/`** folder in your home directory. Copy the skill folder there so Genie Code can use it.

### How to do it

1. In the **workspace file browser** (left sidebar), navigate to the `skill/` folder (path printed in the cell above).
2. Right-click the **`manufacturing-analytics_genie`** folder → **Copy**.
3. Navigate to `/Workspace/Users/<your_email>/.assistant/skills/`.
   - Create `.assistant/` and `skills/` folders if they don't exist (right-click → **Create** → **Folder**).
4. Paste the `manufacturing-analytics_genie` folder there.

**Result:**
```
/Workspace/Users/<your_email>/
  .assistant/
    skills/
      manufacturing-analytics_genie/
        SKILL.md
```

The `_genie` suffix tells Genie Code this is a skill it should load. The skill itself is generic (not tied to a specific catalog or space), so you can reuse it across any manufacturing dataset.

**Docs:** [Where to place skill files](https://docs.databricks.com/en/genie/code/skills.html)

In [ ]:
# Verify the skill is in place — run this after you copy the folder
import os

user_email = w.current_user.me().user_name
expected_path = f"/Workspace/Users/{user_email}/.assistant/skills/manufacturing-analytics_genie/SKILL.md"

if os.path.exists(expected_path):
    size = os.path.getsize(expected_path)
    print(f"Skill found: {expected_path}  ({size:,} bytes)")
    print("Genie Code will use this skill automatically when you open the Assistant.")
else:
    print(f"Skill NOT found at: {expected_path}")
    print()
    print("Follow the instructions in Step 3 above to copy the skill folder.")
    print(f"Source: {skill_source}")
    print(f"Target: /Workspace/Users/{user_email}/.assistant/skills/manufacturing-analytics_genie/SKILL.md")

## Step 4: Use the prompt to BUILD a Genie space via Genie Code

This is the key deliverable — a **specific prompt** that tells Genie Code to generate a notebook which programmatically creates a fully configured Genie SQL space, complete with instructions, sample questions, and curated Q→SQL examples.

The skill provides domain knowledge (what OEE means, how manufacturing data is structured). The prompt provides YOUR specific tables, catalog, and what the space should contain.

### How to use
1. Open a **new, empty notebook**
2. Click the **Assistant** (sparkle) in the sidebar
3. Paste the prompt below
4. Genie Code generates cells that call the Genie REST API to create your space
5. Run the cells — you get a fully configured Genie space

In [ ]:
genie_code_prompt = f"""
Build me a Databricks notebook that programmatically creates a fully configured AI/BI Genie SQL space
using the Genie REST API. Follow the Genie space best practices from the manufacturing-analytics skill
(instructions, samples, curated examples, benchmarks, metadata).

## CRITICAL API RULES (follow these exactly or the API will reject your requests):

### Authentication
Use `w.config.authenticate()` for headers — NEVER use `w.config.token` (it returns None on serverless):
```python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {{**w.config.authenticate(), "Content-Type": "application/json"}}
```

### serialized_space schema (version 2)
The `serialized_space` field is a **JSON string** passed in the POST/PATCH body. This is the ONLY way to set instructions, examples, and sample questions. Do NOT put `sql_instructions`, `sample_questions`, or `curated_questions` as top-level fields in the POST body.

Exact schema:
```json
{{
  "version": 2,
  "config": {{
    "sample_questions": [
      {{"id": "<32-char-hex>", "question": ["<question text>"]}}
    ]
  }},
  "data_sources": {{
    "tables": [
      {{"identifier": "catalog.schema.table"}}
    ]
  }},
  "instructions": {{
    "text_instructions": [{{"content": ["<your SQL instructions text>"]}}],
    "example_question_sqls": [
      {{"id": "<32-char-hex>", "question": ["<question>"], "sql": ["<sql>"]}}
    ]
  }}
}}
```

**Rules:**
- All `id` values: `uuid.uuid4().hex` (32-char lowercase hex)
- `question`, `sql`, `content` are **arrays of strings**, not plain strings
- `example_question_sqls` MUST be **sorted by `id`** or the API returns INVALID_PARAMETER_VALUE
- `data_sources.tables` must also be sorted by `identifier`
- There is NO `evaluation_criteria` field — benchmarks are just additional `example_question_sqls` entries

### Dedup / Idempotency
Before creating a space, list existing spaces and check if one with the same title exists. If found, PATCH it instead of creating a duplicate:
```python
existing = requests.get(f"{{host}}/api/2.0/genie/spaces", headers=headers).json()
for s in existing.get("spaces", []):
    if s.get("title") == my_title:
        # PATCH instead of POST
        break
```

### UI URL
Use `/genie/rooms/` (NOT `/genie/spaces/`) for browser links. For Azure, append `?o=<workspace_id>`:
```python
import re
m = re.search(r"adb-(\d+)\.", host)
o = f"?o={{m.group(1)}}" if m else ""
url = f"{{host}}/genie/rooms/{{space_id}}{{o}}"
```

## My tables (all in {CATALOG}.{SCHEMA}):

| Table | Rows | Key columns |
|-------|------|-------------|
| plants | 8 | plant_id, plant_name, city, state, employee_count, annual_revenue |
| production_lines | 32 | line_id, line_name, plant_id, product_type, daily_capacity, status (Active/Maintenance) |
| operators | 500 | operator_id, first_name, last_name, shift (Morning/Afternoon/Night), certification, plant_id |
| production_events | 50,000 | event_id, event_type, event_date (DATE), production_line_id, operator_id, unit_serial_vin |
| quality_metrics_daily | ~2,200 | date (DATE), plant_id, production_line_id, oee_score (0-1), first_pass_yield (0-1), downtime_minutes, scrap_count, units_produced |
| safety_incidents | 40 | incident_id, severity, category, production_line_id, incident_date |
| equipment_feedback | 100 | feedback_id, equipment_name, comment, production_line_id, operator_id |

## Event types: unit_produced, defect_detected, inspection_passed, downtime_start, scrap, rework_completed

## SQL functions:
- {CATALOG}.{SCHEMA}.compute_defect_rate(line_id, start_date, end_date)
- {CATALOG}.{SCHEMA}.compute_oee_summary(plant_id, start_date, end_date)
- {CATALOG}.{SCHEMA}.get_best_production_line()

## Generate these cells:

### Cell 1: Setup
- `from databricks.sdk import WorkspaceClient`; import requests, json, uuid, re
- `headers = {{**w.config.authenticate(), "Content-Type": "application/json"}}`
- Auto-detect a running SQL warehouse (Pro or Serverless) via `w.warehouses.list()`
- Define catalog = "{CATALOG}", schema = "{SCHEMA}", fqn = f"{{catalog}}.{{schema}}"
- Define TABLE_IDENTIFIERS = sorted list of all 7 tables as f"{{fqn}}.table_name"

### Cell 2: SQL Instructions
Store as a Python string variable `sql_instructions`. Write concise instructions:
- OEE is 0-1 (multiply by 100 for %). FPY same scale.
- Defect rate = COUNT(defect_detected) / COUNT(unit_produced) * 100 from production_events
- Scrap rate = scrap_count / units_produced * 100 from quality_metrics_daily
- Join paths: events->lines (production_line_id=line_id), lines->plants (plant_id), events->operators (operator_id), quality->lines, safety->lines
- Date column is `date` not `metric_date`
- unit_serial_vin is sensitive — use production_events_restricted view for non-admins
- Reference the 3 SQL functions

### Cell 3: 5 Sample Questions
Store as a Python list. Format each as: {{"id": uuid.uuid4().hex, "question": ["<text>"]}}
1. "What is the average OEE by plant for 2024?"
2. "Which production lines have the highest defect rates?"
3. "Show me downtime trends by month for Michigan plants"
4. "How many critical safety incidents happened this year?"
5. "Which shift has the best quality performance?"

### Cell 4: 8 Curated Q->SQL Examples
Store as a Python list. Format each as: {{"id": uuid.uuid4().hex, "question": ["<text>"], "sql": ["<sql>"]}}
Use fully qualified table names ({CATALOG}.{SCHEMA}.table_name) in all SQL.
1. Average OEE % by plant
2. Defect rate by line
3. Total downtime hours by plant 2024
4. Safety incidents by severity
5. Operators by shift at Michigan plants
6. Scrap rate for Michigan, last 5 days of 2024
7. Lines with >5% defect rate (subquery with HAVING)
8. Best quality shift Dec 2024 (MIN defect count)

### Cell 5: 16 Benchmark Questions with Ground Truth SQL
Same format as curated examples: {{"id": uuid.uuid4().hex, "question": ["<text>"], "sql": ["<sql>"]}}
Each SQL must return a single scalar. These will be ADDED to example_question_sqls alongside the 8 curated examples (total: 24).

### Cell 6: Create Space via REST API
1. First, list all spaces and check if "{GENIE_SPACE_PREFIX} - Genie Code Built" already exists
2. If exists: PATCH it. If not: POST to create it.
3. Build serialized_space as a JSON string using the version 2 schema above
4. Include: text_instructions (from Cell 2), sample_questions (Cell 3), example_question_sqls (Cell 4 + Cell 5, sorted by id)
5. Print the Genie UI URL using /genie/rooms/ (not /genie/spaces/)
6. Space title: "{GENIE_SPACE_PREFIX} - Genie Code Built"

### Cell 7: Verify
- GET the space metadata and confirm it was created/updated
- Print summary: title, table count, warehouse
- Print the clickable Genie UI URL using /genie/rooms/

Do NOT create a Cell 7 for "Push Benchmarks" — benchmarks are included in Cell 6 as part of example_question_sqls.
Use the REST API directly (requests library), not SDK genie methods.
"""

print("=" * 70)
print("COPY EVERYTHING BETWEEN THE LINES INTO GENIE CODE")
print("=" * 70)
print(genie_code_prompt)
print("=" * 70)
print(f"\nPrompt length: {len(genie_code_prompt):,} characters")
print()
print("How to use:")
print("  1. Open a NEW empty notebook")
print("  2. Click the Assistant (sparkle icon) in the sidebar")
print("  3. Copy the full prompt above (between the === lines)")
print("  4. Paste it into the Assistant chat box")
print("  5. Genie Code generates 7 cells that build the complete space")
print("  6. Run all cells — you get a fully configured Genie space")

## Step 5: Bonus prompt — Generate a full analysis notebook

This second prompt tells Genie Code to build a complete manufacturing analysis notebook (OEE trends, defect rates, safety dashboard, etc.) using the same domain knowledge from the skill.

### How to use
1. Open another **new, empty notebook**
2. Click the **Assistant** (sparkle) in the sidebar
3. Paste the prompt below
4. Run the generated cells to see a full manufacturing analytics report

In [ ]:
genie_code_prompt = f"""
Create a comprehensive manufacturing quality analysis notebook using the tables in {CATALOG}.{SCHEMA}. 
Use the manufacturing-analytics_genie skill for domain context.

Generate the following analysis sections as separate cells:

1. **Setup** — Set catalog/schema context and show table row counts for all 7 tables

2. **OEE Analysis** — 
   - Average OEE % by plant (use quality_metrics_daily.oee_score * 100)
   - Show which plants are above/below the 85% target
   - Monthly OEE trend chart for 2024

3. **Defect Analysis** —
   - Overall defect rate: COUNT defect_detected / COUNT unit_produced from production_events for 2024
   - Top 10 production lines by defect rate
   - Defect rate by month trend

4. **First Pass Yield** —
   - Average FPY % by plant (quality_metrics_daily.first_pass_yield * 100)
   - Monthly FPY trend

5. **Downtime Analysis** —
   - Total downtime hours by plant for 2024 (downtime_minutes / 60)
   - Top 5 lines with most downtime

6. **Safety Dashboard** —
   - Incident count by severity
   - Incident count by category
   - Monthly incident trend

7. **Operator Insights** —
   - Operator count by shift and plant (join operators to plants)
   - Which shift has the most defect events in Dec 2024 (join production_events to operators)

8. **Executive Summary** —
   - Single cell showing: total units produced, overall defect rate %, average OEE %, total safety incidents, best performing plant by OEE

Use SQL cells where possible. For visualizations, use matplotlib or plotly.
Join tables using: production_events.production_line_id -> production_lines.line_id -> plants.plant_id.
Quality metrics join: quality_metrics_daily.production_line_id -> production_lines.line_id.
Remember: oee_score and first_pass_yield are 0-1 scale (multiply by 100 for %).
"""

print("=" * 70)
print("COPY EVERYTHING BETWEEN THE LINES INTO GENIE CODE")
print("=" * 70)
print(genie_code_prompt)
print("=" * 70)
print(f"\nPrompt length: {len(genie_code_prompt):,} characters")
print()
print("How to use:")
print("  1. Open a NEW empty notebook in your workspace")
print("  2. Click the Assistant (sparkle icon) in the sidebar")
print("  3. Copy the full prompt above (between the === lines)")
print("  4. Paste it into the Assistant chat box")
print("  5. Genie Code generates ~8 cells covering the full analysis")
print("  6. Run all cells — compare results to your Genie SQL space")

## What you learned

| Concept | What you did |
|---------|-------------|
| **Generic skill** | Created a reusable `.md` with manufacturing domain knowledge + Genie space best practices |
| **Skill = domain brain** | Teaches Genie Code what OEE means AND what a good space needs (instructions, examples, benchmarks) |
| **Specific prompt** | Wrote a detailed prompt with YOUR tables, columns, and 16 benchmarks to build a complete space |
| **Prompt = build spec** | Tells Genie Code exactly what to create: instructions + 5 samples + 8 curated Q→SQL + 16 benchmarks |
| **Space-as-code** | One prompt → one notebook → one fully configured space ready for A/B testing and monitoring |

**The pattern:**
```
Generic skill (reusable)          +     Specific prompt (per use case)              =     Output
manufacturing-analytics.md              "Build a space for these 7 tables            Genie SQL space with
 • Domain: metrics, joins, patterns       with 8 examples and 16 benchmarks"           instructions, samples,
 • Best practices: what a good                                                          examples, benchmarks
   space needs                                                                          — ready for production
```

**Next:** Notebook **07** — A/B test your configured vs blank spaces to quantify the impact of curation.

**References:**
- [What is Genie Code?](https://docs.databricks.com/en/genie/code/index.html)
- [Create and manage skills](https://docs.databricks.com/en/genie/code/skills.html)
- [AI/BI Genie spaces](https://docs.databricks.com/en/genie/index.html)
- [Genie REST API reference](https://docs.databricks.com/api/workspace/genie)